# **Análise Exploratória**

Este arquivo é responsável pela análise explorátoria (EDA) dos dados tratados, é por meio dele que irkemos conhecer os dados que estamos trabalhando e quais análises poderemos realizar

---

### **Escopo do Notebook:**

- Criando estilização dos dataframes
- Conectando ao banco de dados
- transformando tabelas em dataframes
- Calculando métricas simples

---

# **Código:**

- **Comentar o código e criar markdowns para cada etapa - amanhã 26/06**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyodbc

### **Criando estilização dos dataframes**

In [2]:
# Meu notebook está formatando as colunas dos datarames tipo object alinhado a direita, por essa razão, este trecho alinha a esquerda

from IPython.display import HTML, display

display(HTML("""
<style>
.dataframe th {
    text-align: left !important;
}

.dataframe td {
    text-align: left !important;
}
</style>
"""))

### **Conectando ao banco de dados**

In [3]:
conn = pyodbc.connect(     
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=monitoramento_db;"
    "Trusted_Connection=yes;"
)

cursor = conn.cursor()

### **transformando principais tabelas em dataframes**

In [4]:
clientes = pd.read_sql("SELECT * FROM clientes", conn)
produtos = pd.read_sql("SELECT * FROM produtos", conn)
centros_distribuicao = pd.read_sql("SELECT * FROM centros_distribuicao", conn)
veiculos = pd.read_sql("SELECT * FROM veiculos", conn)

vw_envio = pd.read_sql("SELECT * FROM vw_envio", conn)
vw_entrega = pd.read_sql("SELECT * FROM vw_entrega", conn)
vw_itens_envio = pd.read_sql("SELECT * FROM vw_itens_envio", conn)

vw_faixa_peso = pd.read_sql("SELECT * FROM vw_faixa_peso", conn)

C:\Users\Igor Cruz\AppData\Local\Temp\ipykernel_27372\1744073776.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  clientes = pd.read_sql("SELECT * FROM clientes", conn)
C:\Users\Igor Cruz\AppData\Local\Temp\ipykernel_27372\1744073776.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  produtos = pd.read_sql("SELECT * FROM produtos", conn)
C:\Users\Igor Cruz\AppData\Local\Temp\ipykernel_27372\1744073776.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  centros_distribuicao = pd.read_sql("SELECT * FROM centros_distribuic

### **Calculando métricas gerais**

In [15]:
num_clientes_cadastrados = vw_envio["cliente"].count() 
num_veiculos_cadastrados = veiculos["id_veiculo"].count()
num_hub_cadastrados = centros_distribuicao["id_cd"].count()
num_total_envios = vw_envio["id_envio"].count()
num_total_produtos = produtos["id_produto"].count()
num_total_status_envios = vw_entrega["id_status_entrega"].value_counts()


envios_por_cliente = vw_envio.groupby("cliente")["id_envio"].count().sort_values(ascending = False).reset_index().rename(columns = { "nome" : "NOME", "id_envio": "QUANTIDADE"})
envios_por_veiculo = vw_envio.groupby("veiculo")["id_envio"].count().sort_values(ascending = False).reset_index().rename(columns = {"modelo": "MODELO", "id_envio": "QUANTIDADE"})
recebidos_por_hub = vw_entrega.groupby("nome_hub")["id_envio"].count().sort_values(ascending = False).reset_index().rename(columns = {"nome_hub": "NOME CD", "id_envio": "QUANTIDADE"})


In [16]:
metricas_basicas = pd.DataFrame({
    "Métrica": [

        "Total de clientes cadastrados",
        "Total de veículos cadastrados",
        "Total de centros de distribuição",
        "Total de envios registrados",
        "Total de produtos cadastrados"

    ],

    "Valor": [

            num_clientes_cadastrados,
            num_veiculos_cadastrados,
            num_hub_cadastrados,
            num_total_envios,
            num_total_produtos

    ]

})

metricas_basicas.to_csv("../data/metricas_basicas.csv", index = False)

### **Criando tabelas csv a partir dos dataframes para consulta rápida e armazenamento na documentação**

In [17]:
envios_por_cliente.to_csv("../data/qtd_envio_por_cliente.csv", index = False)
envios_por_veiculo.to_csv("../data/qtd_envio_por_veiculo.csv", index = True)
recebidos_por_hub.to_csv("../data/qtd_recebido_por_cd.csv", index = True)

In [5]:
vw_envio.to_csv("../data/vw_envio.csv", index = False)

### **Métricas Financeiras**

In [ ]:
# Receita total gerada pelo valor cobrado nos fretes
receita_frete = vw_envio["valor_frete"].sum().round(2)

# Ticket Médio do frete
ticket_medio_frete = vw_envio["valor_frete"].mean().round(2)

# Maior valor de frete
maior_frete = vw_envio["valor_frete"].max()

# Menor valor de frete
menor_frete = vw_envio["valor_frete"].min()

receita_por_cliente = vw_envio.groupby("cliente")["valor_frete"].sum()

# Receita por veículo
receita_por_veiculo = vw_envio.groupby("veiculo")["valor_frete"].sum()

# Receita média por envio
receita_media_por_envio = vw_envio.groupby("id_envio")["valor_frete"].mean()

# Receita por mês
receita_por_mes = vw_envio.groupby(vw_envio["data_postagem"].dt.to_period("M"))["valor_frete"].sum().reset_index().rename(columns = {"data_postagem": "Mês", "valor_frete": "Receita"})



### **Métricas de Cargas**

In [37]:
# Peso total transportado
peso_total_transportado = vw_envio["peso_carga"].sum()

# Peso médio por envio
peso_medio_por_envio = vw_envio.groupby("id_envio")["peso_carga"].mean()

# Maior peso
maior_peso = vw_envio["peso_carga"].max()

# Menor peso
menor_peso = vw_envio["peso_carga"].min()

# Peso por veículo
peso_por_veiculo = vw_envio.groupby("veiculo")["peso_carga"].sum()

# Peso por cliente
peso_por_cliente = vw_envio.groupby("cliente")["peso_carga"].sum()

# Relação Peso × Valor do Frete
peso = (vw_faixa_peso.groupby("faixa_peso", as_index=False)["valor_frete"].sum().sort_values(by="valor_frete", ascending=False))


### **Métricas de Prazo**

In [69]:
# Prazo previsto médio de entrega
prazo_medio_entrega = (vw_envio["data_postagem"] - vw_envio["data_previsao_entrega"]).mean().round("1d")

# Prazo médio real de entrega
prazo_medio_entrega_real = (vw_envio["data_postagem"] - vw_envio["data_entrega_real"]).mean().round("1d")

# Dias médios de atraso
media_dias_atraso = (vw_envio["data_entrega_real"] - vw_envio["data_previsao_entrega"]).mean().round("1d")

# Dias médios de antecipação
media_dias_antecipacao = (vw_envio["data_previsao_entrega"] - vw_envio["data_entrega_real"]).mean().round("1d")

# Percentual de entregas no prazo
percentual_entregas_no_prazo = (vw_envio[vw_envio["data_entrega_real"] <= vw_envio["data_previsao_entrega"]].shape[0] / vw_envio.shape[0]) * 100

# Percentual de entregas atrasadas
percentual_entregas_atrasadas = (vw_envio[vw_envio["data_entrega_real"] > vw_envio["data_previsao_entrega"]].shape[0] / vw_envio.shape[0]) * 100

# Maior atraso registrado
maior_atraso_registrado = (vw_envio["data_entrega_real"] - vw_envio["data_previsao_entrega"]).max().round("1d")

# Clientes com maior atraso médio
clientes_maior_atraso_medio = (vw_envio.groupby("cliente")["data_entrega_real"].mean() - vw_envio.groupby("cliente")["data_previsao_entrega"].mean()).sort_values(ascending=False).reset_index().rename(columns = {"cliente": "Cliente", "data_entrega_real": "Média de atraso"})

# Veículos com maior atraso médio
veiculos_maior_atraso_medio = (vw_envio.groupby("veiculo")["data_entrega_real"].mean() - vw_envio.groupby("veiculo")["data_previsao_entrega"].mean()).sort_values(ascending=False).reset_index().rename(columns = {"veiculo": "Veículo", "data_entrega_real": "Média de atraso"})

# REFAZENDO ALGUMAS MÉTRICAS

# Taxa de entregas concluídas
taxa_entregas_concluidas = (vw_envio[vw_envio["data_entrega_real"].notnull()].shape[0] / vw_envio.shape[0]) * 100

# Taxa de entregas atrasadas
taxa_entregas_atrasadas = (vw_envio[vw_envio["data_entrega_real"] > vw_envio["data_previsao_entrega"]].shape[0] / vw_envio.shape[0]) * 100

# Taxa de entregas antecipadas
taxa_entregas_antecipadas = (vw_envio[vw_envio["data_entrega_real"] < vw_envio["data_previsao_entrega"]].shape[0] / vw_envio.shape[0]) * 100

# Tempo médio de entrega
tempo_medio_entrega = (vw_envio["data_entrega_real"] - vw_envio["data_postagem"]).mean().round("1d")

# SLA (% entregues no prazo)
sla = (vw_envio[vw_envio["data_entrega_real"] <= vw_envio["data_previsao_entrega"]].shape[0] / vw_envio.shape[0]) * 100

# Entregas concluídas por mês
entregas_concluidas_por_mes = vw_envio[vw_envio["data_entrega_real"].notnull()].groupby(vw_envio["data_postagem"].dt.to_period("M"))["id_envio"].count().reset_index().rename(columns = {"data_postagem": "Mês", "id_envio": "Entregas Concluídas"})


### **Métricas de Prazo**

In [70]:
# Receita diária
receita_diaria = vw_envio.groupby(vw_envio["data_postagem"].dt.to_period("D"))["valor_frete"].sum().reset_index().rename(columns = {"data_postagem": "Data", "valor_frete": "Receita"})

# Receita semanal
receita_semanal = vw_envio.groupby(vw_envio["data_postagem"].dt.to_period("W"))["valor_frete"].sum().reset_index().rename(columns = {"data_postagem": "Semana", "valor_frete": "Receita"})

# Receita mensal
receita_mensal = vw_envio.groupby(vw_envio["data_postagem"].dt.to_period("M"))["valor_frete"].sum().reset_index().rename(columns = {"data_postagem": "Mês", "valor_frete": "Receita"})

# Envios por dia da semana
envios_por_dia_semana = vw_envio.groupby(vw_envio["data_postagem"].dt.day_name())["id_envio"].count().reset_index().rename(columns = {"data_postagem": "Dia da Semana", "id_envio": "Quantidade de Envios"})

# Evolução do atraso ao longo dos meses
evolucao_atraso_mensal = (vw_envio.groupby(vw_envio["data_postagem"].dt.to_period("M"))["data_entrega_real"].mean() - vw_envio.groupby(vw_envio["data_postagem"].dt.to_period("M"))["data_previsao_entrega"].mean()).reset_index().rename(columns = {"data_postagem": "Mês", "data_entrega_real": "Média de atraso"})

### **Métricas por Produto**

In [71]:
# Produtos mais enviados
produtos_mais_enviados = vw_itens_envio.groupby("produto")["quantidade"].sum().sort_values(ascending=False).reset_index().rename(columns = {"produto": "Produto", "quantidade": "Quantidade Enviada"})

# Categorias mais enviadas
categorias_mais_enviadas = vw_itens_envio.groupby("categoria_produto")["quantidade"].sum().sort_values(ascending=False).reset_index().rename(columns = {"categoria_produto": "Categoria", "quantidade": "Quantidade Enviada"})

# Quantidade de itens enviados
quantidade_itens_enviados = vw_itens_envio["quantidade"].sum()

# Quantidade média de itens por envio
quantidade_media_itens_por_envio = vw_itens_envio.groupby("id_envio")["quantidade"].mean().reset_index().rename(columns = {"id_envio": "ID do Envio", "quantidade": "Quantidade Média de Itens"})


### **Métricas geográficas**

In [72]:
# ----- CLIENTES ----- 

# Clientes por estado
clientes_por_estado = clientes.groupby("estado")["id_cliente"].count().sort_values(ascending=False).reset_index().rename(columns = {"estado": "Estado", "id_cliente": "Quantidade de Clientes"})

# Clientes por cidade
clientes_por_cidade = clientes.groupby("cidade")["id_cliente"].count().sort_values(ascending=False).reset_index().rename(columns = {"cidade": "Cidade", "id_cliente": "Quantidade de Clientes"})

# Receita por estado
receita_por_estado = vw_envio.groupby("estado_cliente")["valor_frete"].sum().sort_values(ascending=False).reset_index().rename(columns = {"estado_cliente": "Estado", "valor_frete": "Receita"})

# Envios por estado
envios_por_estado = vw_envio.groupby("estado_cliente")["id_envio"].count().sort_values(ascending=False).reset_index().rename(columns = {"estado_cliente": "Estado", "id_envio": "Quantidade de Envios"})

# ----- CENTROS DE DISTRIBUIÇÃO ----- 

# CDs por estado
cds_por_estado = centros_distribuicao.groupby("estado")["id_cd"].count().sort_values(ascending=False).reset_index().rename(columns = {"estado": "Estado", "id_cd": "Quantidade de Centros de Distribuição"})

# Estados que mais recebem entregas
estados_por_recebidos = vw_entrega.groupby("estado")["id_envio"].count().sort_values(ascending=False).reset_index().rename(columns = {"estado": "Estado", "id_envio": "Quantidade de Entregas"})

# Estados que mais enviam cargas
estados_por_envios = vw_envio.groupby("estado_cliente")["id_envio"].count().sort_values(ascending=False).reset_index().rename(columns = {"estado_cliente": "Estado", "id_envio": "Quantidade de Envios"})

### **Métricas por Frota de Veículos**

In [73]:
# Quantidade por categoria
veiculos_por_categoria = veiculos.groupby("categoria")["id_veiculo"].count().sort_values(ascending=False).reset_index().rename(columns = {"categoria": "Categoria", "id_veiculo": "Quantidade de Veículos"})

# Veículos mais utilizados
veiculos_mais_utilizados = vw_envio.groupby("veiculo")["id_envio"].count().sort_values(ascending=False).reset_index().rename(columns = {"veiculo": "Veículo", "id_envio": "Quantidade de Envios"})

